# Overfitting Demo — Polynomial Regression

A **beginner-friendly** walk-through of one of the most important ideas in machine learning: the difference between a model that is **too simple**, a model that is **just right**, and a model that is **too complex**.

The plan:

1. **Make up data** from a known smooth curve, then sprinkle random noise on top (so we know the "truth" the model is trying to recover).
2. **Fit polynomials of increasing degree** — degree 1 is a straight line, degree 15 is a wildly bendy curve.
3. **Look** at a few fits (degree 1, 4, 15) to *see* underfitting vs. a good fit vs. overfitting.
4. **Measure** train error and test error at every degree and plot them together — the famous **U-shaped test curve** appears, and the gap between the two curves is overfitting made visible.

Only `numpy`, `pandas`, `scikit-learn`, and `matplotlib` are used, everything runs offline, and every random step is seeded so you get the same picture every time.

In [ ]:
import numpy as np                                      # numerical arrays + the random-number generator
import pandas as pd                                     # tidy table to summarise errors per degree
import matplotlib.pyplot as plt                         # all plotting
from sklearn.pipeline import make_pipeline              # chain preprocessing + model into one estimator
from sklearn.preprocessing import PolynomialFeatures    # turns x into [1, x, x^2, ... x^d] features
from sklearn.linear_model import LinearRegression       # plain least-squares linear fit on those features
from sklearn.metrics import mean_squared_error          # our error metric: mean squared error (MSE)

# ONE global seed used everywhere below. Fixing it makes the synthetic data, the noise,
# and the train/test split identical on every run -> the plots are perfectly reproducible.
SEED = 42
rng = np.random.default_rng(SEED)   # modern NumPy Generator; we draw all randomness from this object

print("Setup complete. All randomness is seeded with SEED =", SEED)

## 1. Synthesize some data

We invent a **true underlying function** — a smooth curve that the world "really" follows:

$$f(x) = \sin(1.5\pi x)$$

In real life we never observe $f(x)$ directly; every measurement is corrupted by random **noise**. So the data points we actually collect are:

$$y_i = f(x_i) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

where $\varepsilon_i$ is a little bit of Gaussian noise. The whole game of learning is to recover the smooth $f$ **without** chasing the random $\varepsilon$. A model that chases the noise is *overfitting*.

In [ ]:
# --- The true function we're secretly trying to recover ---
def true_function(x):
    """The smooth signal underneath the data. In real life this is unknown; here we control it."""
    return np.sin(1.5 * np.pi * x)

N = 60                                    # how many (x, y) points we collect in total
NOISE_STD = 0.25                          # standard deviation of the Gaussian noise added to each point

# Draw x values uniformly in [0, 1], then SORT them. Sorting isn't needed for fitting,
# but it makes line plots of the fitted curve connect points in left-to-right order.
x = np.sort(rng.uniform(0.0, 1.0, size=N))

# y = smooth truth + random noise. This is the ONLY thing a real model would ever see.
noise = rng.normal(loc=0.0, scale=NOISE_STD, size=N)   # one noise value per point
y = true_function(x) + noise

# sklearn expects the feature matrix X to be 2-D with shape (n_samples, n_features).
# We have a single feature, so reshape the 1-D x into a column vector of shape (N, 1).
X = x.reshape(-1, 1)

print(f"Created {N} noisy data points. X shape = {X.shape}, y shape = {y.shape}")

### Train / test split

We split the points into a **training set** (used to *fit* the model) and a **test set** (kept in a vault, used only to *judge* the model on data it never saw).

This separation is the heart of the demo. A model can memorise the training points to make **training error** look great, but the honest question is: *does it do well on new, unseen points?* Only the **test error** answers that.

In [ ]:
from sklearn.model_selection import train_test_split   # helper that randomly partitions the data

# Hold out 35% of the points as the test set. random_state ties the split to our seed so it
# is identical on every run. We do NOT shuffle-sort afterwards; we re-sort each split below for plotting.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=SEED
)

print(f"Training points: {X_train.shape[0]}")
print(f"Test points:     {X_test.shape[0]}")

### Look at the raw data and the true curve

Before modelling anything, always *look* at your data. Below: the smooth true function (dashed), the training points, and the held-out test points. Notice how the noise scatters the points **around** the true curve — a good model should follow the curve, not weave through every dot.

In [ ]:
# A fine grid of x values spanning the data range, used only to draw smooth curves.
x_grid = np.linspace(0.0, 1.0, 400).reshape(-1, 1)     # shape (400, 1) so sklearn can predict on it

plt.figure(figsize=(8, 5))
# The true underlying function (what we WISH the model would recover).
plt.plot(x_grid, true_function(x_grid), 'k--', linewidth=2, label='true function  f(x) = sin(1.5\u03c0x)')
# Training points (blue) and test points (orange), drawn as scattered dots.
plt.scatter(X_train, y_train, color='tab:blue',   s=40, edgecolor='k', linewidth=0.4, label='train points')
plt.scatter(X_test,  y_test,  color='tab:orange', s=40, edgecolor='k', linewidth=0.4, marker='^', label='test points')
plt.title('Noisy data around a smooth true function')
plt.xlabel('x'); plt.ylabel('y')
plt.legend()
plt.show()

## 2. Polynomial regression

A **degree-$d$ polynomial** model predicts $y$ as a weighted sum of powers of $x$:

$$\hat{y} = w_0 + w_1 x + w_2 x^2 + \dots + w_d x^d$$

This is *still linear regression* — it is linear in the **weights** $w_j$. The trick is to first expand each input $x$ into the features $[1, x, x^2, \dots, x^d]$ (that's what `PolynomialFeatures` does), then let plain `LinearRegression` find the best weights.

The degree $d$ is our **complexity knob**:

- **small $d$** (e.g. 1) → a stiff, simple shape that can't bend enough → **underfitting**.
- **large $d$** (e.g. 15) → an extremely flexible curve that can wiggle through every noisy point → **overfitting**.

We wrap the two steps in a `Pipeline` so "expand features, then fit" behaves like one tidy model.

In [ ]:
def fit_polynomial(degree):
    """Build and train a degree-`degree` polynomial regression on the TRAINING data.

    Returns the fitted pipeline. A Pipeline glues two steps together:
      1. PolynomialFeatures(degree): x  ->  [1, x, x^2, ..., x^degree]
      2. LinearRegression():         fit weights on those expanded features
    Calling .fit / .predict on the pipeline runs both steps in order automatically.
    """
    model = make_pipeline(
        # include_bias=False: we let LinearRegression handle the intercept (w_0) itself,
        # so we don't add a duplicate constant column of 1s in the features.
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )
    model.fit(X_train, y_train)   # learn the weights from the training points only
    return model

# Quick smoke test: fit a degree-3 model and confirm it predicts sensible numbers.
_demo = fit_polynomial(3)
print("Degree-3 model fitted. Prediction at x=0.5:", round(float(_demo.predict([[0.5]])[0]), 3))

### 2a. SEE underfitting vs. good fit vs. overfitting

Let's fit three representative degrees and overlay each fitted curve on the data:

- **Degree 1** — a straight line. Too stiff to follow the sine wave → **underfitting** (high bias).
- **Degree 4** — flexible enough to trace the true curve, but not so flexible it chases noise → a **good fit**.
- **Degree 15** — so flexible it snakes through individual noisy points, especially near the edges → **overfitting** (high variance).

Watch how close each colored curve stays to the black dashed **true function**.

In [ ]:
degrees_to_show = [1, 4, 15]                 # the three illustrative complexity levels

# One row of three side-by-side plots, sharing the same y-axis for fair visual comparison.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, degree in zip(axes, degrees_to_show):
    model = fit_polynomial(degree)                     # train a fresh model of this degree
    y_grid_pred = model.predict(x_grid)                # predict along the fine grid -> smooth fitted curve

    # Report how well it did on train vs test so the label carries the numbers too.
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse  = mean_squared_error(y_test,  model.predict(X_test))

    ax.plot(x_grid, true_function(x_grid), 'k--', linewidth=1.5, label='true function')     # the target
    ax.plot(x_grid, y_grid_pred, color='tab:red', linewidth=2, label=f'degree {degree} fit')  # the model
    ax.scatter(X_train, y_train, color='tab:blue', s=30, edgecolor='k', linewidth=0.3, label='train points')

    # Clip the y-axis: high-degree fits can shoot off to huge values at the edges, which
    # would squash everything else. Limiting the view keeps all three panels readable.
    ax.set_ylim(-2.0, 2.0)
    ax.set_title(f'degree {degree}\ntrain MSE={train_mse:.3f}  |  test MSE={test_mse:.3f}')
    ax.set_xlabel('x')
    ax.legend(fontsize=8, loc='upper right')

axes[0].set_ylabel('y')
plt.suptitle('Underfitting (deg 1)  \u2192  Good fit (deg 4)  \u2192  Overfitting (deg 15)', y=1.05, fontsize=13)
plt.tight_layout()
plt.show()

## 3. Train error vs. test error across all degrees

The three pictures above hint at the story; now let's make it quantitative. We fit a polynomial for **every** degree from 1 to 15 and record two numbers each time:

- **Train MSE** — error on the points the model was fitted on.
- **Test MSE** — error on the held-out points it never saw.

MSE (mean squared error) is the average squared gap between prediction and truth:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - \hat{y}_i\big)^2$$

Smaller is better.

In [ ]:
degrees = range(1, 16)          # try every polynomial degree from 1 up to 15 (inclusive)

train_errors = []               # will hold train MSE for each degree
test_errors  = []               # will hold test  MSE for each degree

for degree in degrees:
    model = fit_polynomial(degree)                          # fit on training data only
    train_errors.append(mean_squared_error(y_train, model.predict(X_train)))  # error on seen data
    test_errors.append(mean_squared_error(y_test,  model.predict(X_test)))    # error on unseen data

# Collect everything into a tidy pandas table for easy reading.
results = pd.DataFrame({
    'degree': list(degrees),
    'train_MSE': train_errors,
    'test_MSE': test_errors,
})

# The best model is the degree with the LOWEST error on the unseen test set.
best_idx = int(np.argmin(test_errors))    # index of the smallest test MSE
best_degree = list(degrees)[best_idx]

print(results.round(4).to_string(index=False))
print(f"\nLowest TEST error is at degree {best_degree} (test MSE = {test_errors[best_idx]:.4f}).")
print(f"Train MSE keeps shrinking from {train_errors[0]:.4f} (deg 1) down to {train_errors[-1]:.4f} (deg 15).")

### The classic overfitting picture

Plotting both error curves against degree reveals the signature shape:

- **Train error (blue)** falls steadily and keeps falling — more flexibility always fits the *training* points better, all the way to near-zero.
- **Test error (orange)** forms a **U**: it drops as the model gets useful, bottoms out at the sweet spot, then climbs again as the model starts memorising noise.

The **growing gap** between the two curves on the right-hand side *is* overfitting. The green marker flags the degree with the lowest test error — the best complexity for generalising to new data.

In [ ]:
plt.figure(figsize=(9, 5.5))

# Both curves on ONE axes so the gap between them is directly visible.
plt.plot(list(degrees), train_errors, 'o-', color='tab:blue',   label='train MSE')
plt.plot(list(degrees), test_errors,  's-', color='tab:orange', label='test MSE')

# Mark the winning degree (lowest test error) with a big green star + a vertical guide line.
plt.scatter([best_degree], [test_errors[best_idx]], color='tab:green', s=220, marker='*',
            zorder=5, label=f'best degree = {best_degree}')
plt.axvline(best_degree, color='tab:green', linestyle=':', alpha=0.6)

# A log y-scale keeps the tiny train errors and the large test errors both visible in one view.
plt.yscale('log')
plt.xlabel('polynomial degree  (model complexity \u2192)')
plt.ylabel('mean squared error  (log scale)')
plt.title('Train error keeps falling, test error is U-shaped \u2014 the gap is overfitting')
plt.xticks(list(degrees))
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

## 4. Why this happens — the bias-variance trade-off

Every model's expected error on new data can be decomposed into three pieces:

$$\underbrace{\text{Error}}_{\text{test error}} \;=\; \underbrace{\text{Bias}^2}_{\text{too simple}} \;+\; \underbrace{\text{Variance}}_{\text{too complex}} \;+\; \underbrace{\text{Irreducible noise}}_{\sigma^2}$$

**Underfitting = high bias.** A degree-1 line is too rigid to represent a sine wave. It makes systematic mistakes no matter how much data you give it. Both train and test error stay high — the model is simply *wrong about the shape*.

**Overfitting = high variance.** A degree-15 polynomial is so flexible it fits the random noise, not just the signal. It nails the training points (train error → ~0) but the wild wiggles are specific to *this* noisy sample; on new points they are wrong, so test error explodes. Re-draw the noise and the fitted curve would swing to a totally different shape — that sensitivity *is* variance.

**The sweet spot** (here, the degree marked in green) balances the two: flexible enough to capture the true curve, stiff enough to ignore the noise.

### Why train error is a misleading gauge

Train error is **optimistic**: the model was tuned to minimise exactly that number, so it can always be driven down by adding complexity — even down to zero if the model can interpolate every point. That tells you *nothing* about new data. This is precisely why we keep a **held-out test set** (and, in practice, use cross-validation): the only honest measure of generalisation is performance on data the model did not learn from.

### Connection to regularization

We controlled complexity here by picking the polynomial **degree**. A more common lever is **regularization** — keeping a high-degree model but adding a penalty on large weights (Ridge / L2: $\lambda\sum w_j^2$, or Lasso / L1: $\lambda\sum |w_j|$). The penalty discourages the extreme coefficients that produce those wild wiggles, pulling a would-be overfit model back toward a smoother fit. Whether you tune the degree, add a regularization penalty, or gather more data, the goal is the same: **land at the bottom of the U-shaped test-error curve.**

## 5. Takeaways

- **Underfitting (high bias):** model too simple → high train *and* test error (degree 1).
- **Good fit:** captures the signal, ignores the noise → lowest test error (the green-starred degree).
- **Overfitting (high variance):** model too complex → train error tiny, test error large; the **gap** between the curves is the tell.
- **Never trust train error alone** — always judge on held-out / cross-validated data.
- Manage complexity by choosing model capacity, using **regularization**, or getting more data.